# Control de Calidad y Filtrado de Noticias (`qa_filtrado.ipynb`)

Este cuaderno realiza un control de calidad sobre el JSON de noticias ya limpiado (`noticias_limpio.json`).

### ¿Qué detecta?
1. **Contenido vacío o nulo.**
2. **Contenido con muy pocas palabras** (por debajo de un umbral configurable), típico de artículos truncados por paywalls.
3. **Páginas de error o bloqueo de scraping** (Cloudflare, *Access Denied*, Captchas, etc.) que se hayan colado como contenido.

### Salidas:
- `noticias_eliminadas.json`: Elementos descartados con los campos extra `motivo_eliminacion` y `num_palabras`.
- `noticias_validas.json`: Elementos limpios e intactos.

## 1. Carga del archivo JSON

Sube directamente el archivo de noticias ya limpiado por la Fase 1
(`noticias_limpio.json` o el nombre que le hayas dado) — sin necesidad de tenerlo
previamente en el entorno de Colab. Los nombres de los archivos de salida se derivan
automáticamente a partir del nombre del archivo subido.


In [ ]:
# --- Detección de entorno (Colab vs. local) ----------------------------------
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("No estamos en Google Colab: se omite el selector de subida.")

import os

ARCHIVO_ENTRADA = None

if IN_COLAB:
    print("Sube tu archivo JSON de noticias ya limpiado (Fase 1):")
    subido = files.upload()
    if subido:
        ARCHIVO_ENTRADA = list(subido.keys())[0]
        print(f"Archivo recibido: {ARCHIVO_ENTRADA}")
    else:
        raise RuntimeError("No se subió ningún archivo. Vuelve a ejecutar la celda para intentarlo de nuevo.")
else:
    # Fuera de Colab: usa un archivo local ya presente en el directorio actual.
    ARCHIVO_ENTRADA = "noticias_limpio.json"
    if not os.path.exists(ARCHIVO_ENTRADA):
        raise FileNotFoundError(
            f"No se encontró '{ARCHIVO_ENTRADA}' localmente. Coloca tu archivo de entrada "
            f"en el directorio actual o ajusta la variable ARCHIVO_ENTRADA manualmente."
        )
    print(f"Usando archivo local: {ARCHIVO_ENTRADA}")

# Los archivos de salida se derivan del nombre del de entrada, para no
# sobrescribirlo y para que quede claro a qué lote pertenecen.
_nombre_base, _ext = os.path.splitext(ARCHIVO_ENTRADA)
if not _ext:
    _ext = ".json"
ARCHIVO_ELIMINADAS = f"{_nombre_base}_eliminadas{_ext}"
ARCHIVO_VALIDAS = f"{_nombre_base}_validas{_ext}"

print(f"Salida (eliminadas): {ARCHIVO_ELIMINADAS}")
print(f"Salida (válidas):    {ARCHIVO_VALIDAS}")


## 2. Función core de control de calidad

In [ ]:
import json
import re
from pathlib import Path

# Umbral por defecto de palabras mínimas
MIN_PALABRAS_DEFECTO = 50

# Patrones de páginas de error / bloqueo de scraping
_PATRONES_ERROR = [
    r"A required part of this site couldn.t load",
    r"Enable JavaScript and cookies to continue",
    r"Please enable JavaScript",
    r"Access Denied",
    r"403 Forbidden",
    r"404 Not Found",
    r"Please verify you are a human",
    r"verify you are human",
    r"[Cc]aptcha",
    r"Client Challenge",
    r"Checking your browser before accessing",
    r"This page isn.t available|Page not found",
    r"Just a moment\.\.\.",
]
_REGEX_ERROR = re.compile("|".join(_PATRONES_ERROR), re.IGNORECASE)

In [ ]:
def contar_palabras(texto: str) -> int:
    """Cuenta palabras separando por espacios en blanco."""
    if not texto:
        return 0
    return len(texto.split())

def evaluar_elemento(item: dict, min_palabras: int) -> tuple[bool, str | None, int]:
    """Decide si un elemento debe eliminarse. Devuelve (eliminar, motivo, num_palabras)."""
    contenido = item.get("contenido", "") or ""
    num_palabras = contar_palabras(contenido)

    if not contenido.strip():
        return True, "contenido_vacio", num_palabras

    if _REGEX_ERROR.search(contenido) or _REGEX_ERROR.search(item.get("titulo", "") or ""):
        return True, "pagina_de_error_o_bloqueo", num_palabras

    if num_palabras < min_palabras:
        return True, f"contenido_demasiado_corto(<{min_palabras}_palabras)", num_palabras

    return False, None, num_palabras

In [ ]:
def ejecutar_qa(
    input_file: str = "noticias_limpio.json",
    output_eliminadas: str = "noticias_eliminadas.json",
    output_validas: str = "noticias_validas.json",
    min_palabras: int = MIN_PALABRAS_DEFECTO,
) -> bool:
    ruta = Path(input_file)
    if not ruta.exists():
        print(f"Error: no se encuentra el archivo {input_file}")
        return False

    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)

    eliminadas, validas = [], []
    conteo_motivos: dict[str, int] = {}

    for item in data:
        eliminar, motivo, num_palabras = evaluar_elemento(item, min_palabras)
        if eliminar:
            registro = dict(item)
            registro["motivo_eliminacion"] = motivo
            registro["num_palabras"] = num_palabras
            eliminadas.append(registro)
            conteo_motivos[motivo] = conteo_motivos.get(motivo, 0) + 1
        else:
            validas.append(item)

    with open(output_eliminadas, "w", encoding="utf-8") as f:
        json.dump(eliminadas, f, ensure_ascii=False, indent=2)

    with open(output_validas, "w", encoding="utf-8") as f:
        json.dump(validas, f, ensure_ascii=False, indent=2)

    print(f"Total analizado:      {len(data)}")
    print(f"Elementos eliminados: {len(eliminadas)}")
    print(f"Elementos válidos:    {len(validas)}")
    print("\nDesglose por motivo de eliminación:")
    for motivo, n in sorted(conteo_motivos.items(), key=lambda x: -x[1]):
        print(f"  - {motivo}: {n}")
    print(f"\nGuardado: {output_eliminadas}")
    print(f"Guardado: {output_validas}")

    return True

## 3. Ejecución del script

Usa directamente el archivo subido en la Sección 1 (`ARCHIVO_ENTRADA`) y los nombres
de salida ya calculados (`ARCHIVO_ELIMINADAS` / `ARCHIVO_VALIDAS`). Ajusta solo
`UMBRAL_PALABRAS` si quieres ser más o menos estricto.


In [ ]:
UMBRAL_PALABRAS = 50  # Cambia este número si quieres ser más o menos estricto

# Ejecutar el proceso sobre el archivo subido en la Sección 1
ejecutar_qa(
    input_file=ARCHIVO_ENTRADA,
    output_eliminadas=ARCHIVO_ELIMINADAS,
    output_validas=ARCHIVO_VALIDAS,
    min_palabras=UMBRAL_PALABRAS,
)


## 4. Descarga de los resultados

Descarga a tu ordenador los dos archivos generados: las noticias que pasaron el
control de calidad (`ARCHIVO_VALIDAS`) y las que fueron descartadas, con su motivo
(`ARCHIVO_ELIMINADAS`).


In [ ]:
if IN_COLAB:
    print(f"Descargando {ARCHIVO_VALIDAS} ...")
    files.download(ARCHIVO_VALIDAS)

    print(f"Descargando {ARCHIVO_ELIMINADAS} ...")
    files.download(ARCHIVO_ELIMINADAS)
else:
    print("Fuera de Colab: los archivos ya están guardados en el directorio actual; "
          f"descárgalos directamente desde ahí ({ARCHIVO_VALIDAS}, {ARCHIVO_ELIMINADAS}).")
